In [0]:
from pyspark.sql import functions as F

df = spark.table("personal_projects.information_retail_schema.online_retail_raw")

df_clean = (df
    .filter(F.col("Quantity") > 0)               # drop returns/cancellations
    .filter(F.col("Price") > 0)
    .filter(F.col("Customer_ID").isNotNull())      # drop anonymous transactions
    .dropDuplicates(["Invoice", "StockCode", "Customer_ID"])
    .withColumn("InvoiceDate", F.to_timestamp("InvoiceDate", "M/d/yyyy H:mm"))
    .withColumn("Revenue", F.col("Quantity") * F.col("Price"))
    .withColumnRenamed("Customer_ID", "CustomerID")
)

df_clean.write.format("delta").mode("overwrite").saveAsTable("personal_projects.information_retail_schema.silver_online_retail_clean")